# STAC data prep — GLiM (GeoParquet)

Converts the GLiM shapefile to a single **GeoParquet** file and writes the enriched metadata next to it.

- **Case:** vector data, one layer, 1.2 million polygons with 6 attribute columns, no time dimension (~2.6 GB .shp + 0.7 GB .dbf as shapefile). Source CRS: Web Mercator (EPSG:3857).
- **Format:** one GeoParquet (see `DATA_FORMATS.md`, vector row; same approach as `18_hydrolakes` and `19_country_boundaries`). Rows are sorted along a Hilbert curve and a bbox covering column is written, so readers can filter spatially without loading everything.
- **OGC service:** not created. The GeoServer of the Global Coastal Atlas has no GeoParquet data store, and the shapefile is not published instead.
- **Metadata template:** `metadata/metadata_GLiM.json`. It is incomplete (empty bbox, time, keywords, media type, collection id), so the missing fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\GLIM_EPSG3857.shp`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1 -Notebook 23_glim.ipynb -Variable GLIM_EPSG3857` from the host. It copies the shapefile parts to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror. The notebook needs about 8 GB of memory.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "GLIM_EPSG3857"
COLLECTION_ID = "glim"
PARQUET_NAME = "glim"

# DATA_VIVIAN is set in docker, where the P: drive is not mounted (a local mirror is used and stac/ is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_shp = data_vivian / f"{VARIABLE}.shp"
metadata_template_path = stac_repo / "metadata" / "metadata_GLiM.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
parquet_dir = processed_data_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

for p in (source_shp, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_shp)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\GLIM_EPSG3857.shp
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_GLiM.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\glim


### Read raw data

Takes a few minutes and about 8 GB of memory.

In [3]:
gdf = gpd.read_file(source_shp, engine="pyogrio", encoding="utf-8")

empty = gdf.geometry.isna() | gdf.geometry.is_empty
print("rows:", len(gdf), "| crs:", gdf.crs.to_string() if gdf.crs else None)
print("geometry types:", gdf.geom_type.value_counts().to_dict())
print("OBJECTID unique:", gdf["OBJECTID"].is_unique, "| null or empty geometries:", int(empty.sum()))
print("total bounds:", gdf.total_bounds.tolist())
gdf.drop(columns="geometry").head()

rows: 1235259 | crs: EPSG:3857
geometry types: {'Polygon': 1229262, 'MultiPolygon': 5997}
OBJECTID unique: True | null or empty geometries: 0
total bounds: [-20037507.06750193, -57137565.40378635, 20037507.067577902, 18477721.353954516]


,OBJECTID,IDENTITY_,Litho,xx,Shape_Leng,Shape_Area
0,1.0,FJI232,vi____,vi,708.487484,34086.593471
1,2.0,FJI233,vi____,vi,713.647900,29973.267410
2,3.0,FJI234,vi____,vi,289.131532,6157.096080
3,4.0,FJI235,vi____,vi,369.092971,9331.273820
4,5.0,FJI236,vi____,vi,285.277658,5283.623787


### Check the CRS

The shapefile is in Web Mercator (EPSG:3857). The STAC catalog uses EPSG:4326, so the geometries are reprojected. Attributes are not touched: the `Shape_*` length/area fields keep the values of the original CRS.

In [4]:
print("CRS before:", gdf.crs.to_string() if gdf.crs else None)
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
print("CRS:", gdf.crs.to_string())
print("total bounds (EPSG:4326):", gdf.total_bounds.tolist())

CRS before: EPSG:3857
CRS: EPSG:4326
total bounds (EPSG:4326): [-179.99998854389912, -89.98525679540772, 179.9999885445816, 83.68252386080846]


### Make the metadata complete (dataset dependent)

Fill the fields that are empty in the template. Column descriptions follow the GLiM paper (Hartmann and Moosdorf, 2012); please double-check the descriptions that are marked with `?` before publishing.

In [5]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

# Descriptions per attribute column (used as table:columns in the STAC item)
table_columns = {
    "OBJECTID": "Feature identifier",
    "IDENTITY_": "Polygon identifier (region code + number, e.g. FJI232)",
    "Litho": "Lithology code: three 2-letter levels (main class, subclass, additional characteristics); '_' = not available",
    "xx": "First level lithological class (2-letter code, first level of Litho)",
    "Shape_Leng": "Polygon perimeter in the original EPSG:3857 CRS (m, not updated after reprojection)",
    "Shape_Area": "Polygon area in the original EPSG:3857 CRS (m2, not updated after reprojection)",
    "geometry": "Lithological unit polygon (EPSG:4326)",
}
missing = set(gdf.columns) - set(table_columns)
assert not missing, f"Columns without description: {missing}"

west, south, east, north = (float(v) for v in gdf.total_bounds)
bbox = [max(west, -180.0), max(south, -90.0), min(east, 180.0), min(north, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "GLiM",
    "HISTORY": [
        "GLiM v1.0 (Hartmann and Moosdorf, 2012), shapefile GLIM_EPSG3857.shp (Web Mercator)",
        "Reprojected to EPSG:4326 and converted to a single GeoParquet (zstd, rows sorted along a Hilbert curve, bbox covering column)",
        "Attributes are unchanged (the Shape_Leng and Shape_Area fields still hold the values of the original EPSG:3857 geometries)"
],
    "MEDIA_TYPE": "application/vnd.apache.parquet",
    "DATA_MODEL": "vector_geoparquet",
    "DIMENSIONS": [],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    # Publication year; the dataset has no time dimension
    "TEMPORAL_EXTENT": ["2012-01-01T00:00:00Z", ""],
    "KEYWORDS": ["lithology", "geology", "rock types", "geochemistry", "global"],
    "TAGS": ["lithology", "geology", "polygons", "geoparquet"],
    "LONG_NAME": "Global lithological map polygons",
    "UNITS": "",
    "COMMENT": (
        "Vector layer with one polygon per lithological unit. The Litho code has three 2-letter levels; the first "
        "level (xx) is the main lithological class (e.g. su = unconsolidated sediments, ss = siliciclastic "
        "sedimentary, sc = carbonate sedimentary, mt = metamorphics, pa/pi/pb = acid/intermediate/basic plutonic, "
        "va/vi/vb = acid/intermediate/basic volcanic, py = pyroclastics, ev = evaporites, wb = water bodies, ig = ice "
        "and glaciers, nd = no data). The temporal extent is the publication year; the layer is static."
    ),
    "AUTHOR": "Hartmann J, Moosdorf N",
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": None,
    "NODATA": None,
    "DATA_TYPE": "GeoParquet",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": "",  # no OGC service: GeoServer has no GeoParquet data store
    "FEATURE_COUNT": int(len(gdf)),
    "TABLE_COLUMNS": table_columns,
})

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("FEATURE_COUNT:", metadata["FEATURE_COUNT"])

SPATIAL_EXTENT: [-179.999989, -89.985257, 179.999989, 83.682524]
FEATURE_COUNT: 1235259


### Write data to GeoParquet

`zstd` compression, row groups of 100 000 features, rows ordered along a Hilbert curve so that nearby features share a row group, and a `bbox` covering column for spatial filtering. Null or empty geometries (if any) are put at the end, because they have no position on the curve.

In [6]:
valid = ~(gdf.geometry.isna() | gdf.geometry.is_empty).to_numpy()
hilbert = gdf.geometry[valid].hilbert_distance().to_numpy()
order = np.concatenate([np.flatnonzero(valid)[np.argsort(hilbert)], np.flatnonzero(~valid)])
gdf = gdf.iloc[order].reset_index(drop=True)

out_path = parquet_dir / f"{PARQUET_NAME}.parquet"
gdf.to_parquet(
    out_path,
    index=False,
    compression="zstd",
    row_group_size=100_000,
    write_covering_bbox=True,
)

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

before_mb = sum(p.stat().st_size for p in source_shp.parent.glob(f"{VARIABLE}.*")) / 1e6
print(f"Wrote GeoParquet: {out_path} ({out_path.stat().st_size / 1e6:.0f} MB, shapefile parts {before_mb:.0f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote GeoParquet: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\glim\parquet\glim.parquet (2157 MB, shapefile parts 3483 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\glim\metadata_glim.json


### Verify the output

Same rows (in the same order), columns and values as the processed GeoDataFrame, EPSG:4326, readable as GeoParquet.

In [7]:
meta = pq.ParquetFile(out_path).metadata
print("row groups:", meta.num_row_groups, "| rows:", meta.num_rows)
assert meta.num_rows == len(gdf)

new = gpd.read_parquet(out_path)
assert new.crs.to_epsg() == 4326
assert list(new.columns.drop("bbox", errors="ignore")) == list(gdf.columns)

# Rows were written in the order of gdf, so they can be compared position by position
attrs = [c for c in gdf.columns if c != "geometry"]
pd.testing.assert_frame_equal(pd.DataFrame(new[attrs]), pd.DataFrame(gdf[attrs]))
same_geom = new.geometry.geom_equals_exact(gdf.geometry, tolerance=0) | (new.geometry.isna() & gdf.geometry.isna())
assert same_geom.all(), "Geometries changed"
print("OK: glim GeoParquet and metadata are ready for the STAC builder notebook.")

row groups: 13 | rows: 1235259
OK: glim GeoParquet and metadata are ready for the STAC builder notebook.
